# Qwen3-1.7B safety post-training on one Kaggle T4
This notebook orchestrates the repository scripts. Public-source labels are mapped automatically to a binary safe/unsafe taxonomy, and real training is opt-in.

In [ ]:
import os, subprocess, sys
# The experiment is designed for one T4. Kaggle may allocate two, which makes
# Transformers use DataParallel and breaks bitsandbytes 4-bit training.
os.environ['CUDA_VISIBLE_DEVICES'] = '0'
subprocess.run([sys.executable, '-c', "import torch; print('torch', torch.__version__, 'CUDA', torch.version.cuda); print('visible GPU count', torch.cuda.device_count()); print('GPU', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'none')"], check=True)

## Repository and compatible dependencies
Enable Internet and a GPU. The first cell intentionally exposes only GPU 0 because this experiment uses one T4, even when Kaggle allocates two. Add a Kaggle secret named `token-30-days` containing a GitHub token with read access to the private repository, and grant this notebook permission to use it. The cell below authenticates only for the clone/pull operation; it does not put the token in the URL or save it as the Git remote.

In [ ]:
import base64, os
from pathlib import Path
from kaggle_secrets import UserSecretsClient

REPO = Path('/kaggle/working/black-box-diff-exploration')
REPO_URL = 'https://github.com/brazhou04/black-box-diff-exploration.git'
token = UserSecretsClient().get_secret('token-30-days')
assert token, 'Kaggle secret token-30-days is missing or empty'
basic_auth = base64.b64encode(f'x-access-token:{token}'.encode()).decode()
git_env = os.environ.copy()
git_env.update({
    'GIT_CONFIG_COUNT': '1',
    'GIT_CONFIG_KEY_0': 'http.https://github.com/.extraheader',
    'GIT_CONFIG_VALUE_0': f'Authorization: Basic {basic_auth}',
})
try:
    if (REPO / '.git').is_dir():
        subprocess.run(['git', '-C', str(REPO), 'pull', '--ff-only', 'origin', 'main'], env=git_env, check=True)
    elif REPO.exists():
        raise RuntimeError(f'{REPO} exists but is not a Git repository')
    else:
        subprocess.run(['git', 'clone', REPO_URL, str(REPO)], env=git_env, check=True)
finally:
    del token, basic_auth, git_env

os.chdir(REPO)
os.environ['HF_HOME'] = '/kaggle/working/hf_cache'
print('Repository ready:', REPO)

In [ ]:
%pip install --upgrade -q -r requirements-kaggle.txt
!python scripts/verify_dependencies.py

If verification still reports an old imported package, restart the Kaggle kernel once, rerun the repository cell, and rerun the verifier.

## Fixture preflight and one-step smoke test
This readiness check uses bundled synthetic fixtures, so it does not expect the real study data yet. Smoke artifacts are isolated under the `_smoke` namespace.

In [ ]:
subprocess.run([sys.executable, 'scripts/kaggle_preflight.py', '--smoke-test'], check=True)
subprocess.run([sys.executable, 'run_training_matrix.py', '--conditions', 'M1', 'M2', 'M3', '--seeds', '42', '--smoke-test', '--resume'], check=True)
for smoke_condition in ('M1', 'M2', 'M3'):
    subprocess.run([sys.executable, 'evaluate_safety.py', '--condition', smoke_condition, '--seed', '42', '--smoke-test'], check=True)

## Acquire and finalize binary source-trust data
This downloads revision-pinned records from UltraChat 200k, PKU-SafeRLHF, HarmBench, and XSTest, then automatically creates the binary safe/unsafe train and evaluation files. It trusts the documented source labels without independent human review and makes no separate dual-use claim. Review source licenses first; PKU-SafeRLHF is non-commercial.

In [ ]:
import json
from safety_training.io import sha256_file

prepared_manifest = REPO / 'data/preparation_manifest.json'
prepared = json.loads(prepared_manifest.read_text()) if prepared_manifest.exists() else {}
prepared_outputs = prepared.get('outputs', {})
prepared_ready = (
    prepared.get('preparation_mode') == 'binary_source_trust_without_human_review'
    and bool(prepared_outputs)
    and all(Path(path).is_file() and sha256_file(path) == entry['sha256'] for path, entry in prepared_outputs.items())
)
if prepared_ready:
    print('Reusing manifest-verified binary source-trust data:', prepared_manifest)
else:
    candidate_manifest = REPO / 'data/review/source_manifest.json'
    if not candidate_manifest.exists():
        subprocess.run([sys.executable, 'scripts/acquire_public_data.py', '--train-examples', '300', '--eval-examples', '100'], check=True)
    else:
        print('Reusing existing downloaded candidate pool:', candidate_manifest)
    finalize_command = [sys.executable, 'scripts/finalize_trusted_data.py']
    if prepared_manifest.exists():
        finalize_command.append('--overwrite')
        print('Replacing incomplete or stale prepared data.')
    subprocess.run(finalize_command, check=True)

The automatic mapping uses filtered UltraChat turns as safe examples; PKU pairs with exactly one source-labeled safe response as unsafe training prompts and safe targets; HarmBench behaviors as unsafe evaluation prompts; and source-safe XSTest prompts for overrefusal. Ambiguous cases are not independently identified, so record that limitation in any report. The committed recovery snapshot already contains the completed M3 targets and frozen evaluation suite.

In [ ]:
# The manifest-verified recovery snapshot already contains the expensive M3 cache.
# Do not regenerate it unless deliberately designing a new experiment.
constitutional_target = REPO / 'data/safety_constitutional/train.jsonl'
constitutional_manifest = REPO / 'data/safety_constitutional/generation_manifest.json'
assert constitutional_target.is_file() and constitutional_manifest.is_file()
subprocess.run([sys.executable, 'scripts/validate_experiment.py'], check=True)
# !python scripts/check_experimental_balance.py --model-tokenizer
# !python scripts/kaggle_preflight.py

## Selected real training job
Set `RUN_SELECTED_JOB=True` only after the real-data preflight prints `READY`. Change the condition and seed deliberately.

In [ ]:
RUN_SELECTED_JOB = False
CONDITION = 'M2'
SEED = 42
CONFIGS = {'M1': 'configs/m1_benign.yaml', 'M2': 'configs/m2_safety_sft.yaml', 'M3': 'configs/m3_constitutional.yaml'}
assert CONDITION in CONFIGS and SEED in (42, 123, 456)
if RUN_SELECTED_JOB:
    subprocess.run([sys.executable, 'create_m0_manifest.py'], check=True)
    subprocess.run([sys.executable, 'train.py', '--config', CONFIGS[CONDITION], '--seed', str(SEED), '--resume'], check=True)
else:
    print('Real training is disabled. Complete source-trust preparation/preflight, then set RUN_SELECTED_JOB=True.')

## Immediate frozen audit
The built-in heuristic verifies the pipeline only. Configure a validated `SafetyScorer` for research conclusions.

In [ ]:
if RUN_SELECTED_JOB:
    subprocess.run([sys.executable, 'evaluate_safety.py', '--condition', CONDITION, '--seed', str(SEED)], check=True)

## Artifact summary and optional matrix

In [ ]:
!python run_training_matrix.py --conditions M1 M2 M3 --seeds 42 123 456 --dry-run
# Run the full matrix only when intended:
# !python run_training_matrix.py --conditions M1 M2 M3 --seeds 42 123 456 --resume
# Optional M4 after its matching M3:
# !python train_dpo.py --config configs/m4_dpo.yaml --seed 42 --resume